# Read me

This notebook uses a NearestNeighbors model with Euclidean distance to recommend 20 tracks based on the profile of 10 input tracks. The profile is built by calculating the mean of audio features of the input tracks. 

In [1]:
import pandas as pd

from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import NearestNeighbors

# Load dataset

In [46]:
df = pd.read_csv("./raw_data/spotify-tracks-dataset-detailed.csv")
print(f'Loaded {len(df)} rows')

Loaded 114000 rows


# Cleaning dataset
## Removing Comedy genre

In [47]:
len_before = len(df)
df = df[df["track_genre"] != "comedy"].reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} comedy tracks. Left {len_after} rows')

Removed 1000 comedy tracks. Left 113000 rows


## Droping duplicates

In [49]:
len_before = len(df)
df = df.drop_duplicates(subset=["track_id"]).drop_duplicates(subset=["artists", "track_name"]).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} duplicates. Left {len_after} rows')

Removed 32642 duplicates. Left 80358 rows


## Missing values

In [50]:
len_before = len(df)
df = df.dropna(subset=['album_name']).dropna(subset=['track_name']).reset_index(drop=True)
len_after = len(df)

print(f'Removed {len_before - len_after} null values. Left {len(df)} rows')

Removed 1 null values. Left 80357 rows


## Scaling

In [51]:
audio_features = [
    "danceability",
    "energy",
    "loudness",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo"
]

scaler = StandardScaler().set_output(transform="pandas")

X = df[audio_features]

X_scaled = scaler.fit_transform(X)

In [6]:
X_scaled.head()

,danceability,energy,loudness,speechiness,acousticness,instrumentalness,liveness,valence,tempo
0,0.628394,-0.721328,0.298800,0.551483,-0.849230,-0.503538,0.758094,0.928984,-1.142994
1,-0.847891,-1.896382,-1.794228,-0.079182,1.837684,-0.503524,-0.591264,-0.799395,-1.490909
2,-0.744089,-1.127618,-0.297440,-0.273960,-0.313535,-0.503542,-0.507257,-1.366520,-1.529510
3,-1.735968,-2.320198,-2.049645,-0.457392,1.780439,-0.503313,-0.428501,-1.277786,1.987275
4,0.293923,-0.793026,-0.286864,-0.303272,0.466809,-0.503542,-0.686296,-1.185194,-0.074292


In [7]:
df.head()

,track_id,artists,album_name,track_name,popularity,duration_ms,explicit,danceability,energy,key,loudness,mode,speechiness,acousticness,instrumentalness,liveness,valence,tempo,time_signature,track_genre
0,5SuOikwiRyPMVoIQDJUgSV,Gen Hoshino,Comedy,Comedy,73,230666,False,0.676,0.4610,1,-6.746,0,0.1430,0.0322,0.000001,0.3580,0.715,87.917,4,acoustic
1,4qPNDBW1i3p13qLCt0Ki3A,Ben Woodward,Ghost (Acoustic),Ghost - Acoustic,55,149610,False,0.420,0.1660,1,-17.235,1,0.0763,0.9240,0.000006,0.1010,0.267,77.489,4,acoustic
2,1iJBSr7s7jYXzM8EGcbK5b,Ingrid Michaelson;ZAYN,To Begin Again,To Begin Again,57,210826,False,0.438,0.3590,0,-9.734,1,0.0557,0.2100,0.000000,0.1170,0.120,76.332,4,acoustic
3,6lfxq3CG4xtTiEg7opyCyx,Kina Grannis,Crazy Rich Asians (Original Motion Picture Sou...,Can't Help Falling In Love,71,201933,False,0.266,0.0596,0,-18.515,1,0.0363,0.9050,0.000071,0.1320,0.143,181.740,3,acoustic
4,5vjLSffimiIP26QG5WcN2K,Chord Overstreet,Hold On,Hold On,82,198853,False,0.618,0.4430,2,-9.681,1,0.0526,0.4690,0.000000,0.0829,0.167,119.949,4,acoustic


# Model 1
- NearestNeighbors with Euclidean distance
- Input tracks: Average acoustic profile (Centroid k-NN)

## Train

In [52]:
# NearestNeighbors with Euclidean distance
model = NearestNeighbors(metric="euclidean")

model.fit(X_scaled)

,n_neighbors,5
,radius,1.0
,algorithm,'auto'
,leaf_size,30
,metric,'euclidean'
,p,2
,metric_params,None
,n_jobs,None


## Input tracks

In [53]:
# Build an input track set of 10 tracks
input_tracks = X_scaled.sample(n=10, random_state=42)

In [57]:
# Print input track names
input_indices = input_tracks.index.to_list()
df.iloc[input_indices][['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
48729,6I2tqFhk8tq69iursYxuxd,SEVENTEEN,SEVENTEEN 4th Album 'Face the Sun',HOT,k-pop
43503,6dBUzqjtbnIa1TwYbyw5CM,TV Girl,French Exit,Lovers Rock,indie-pop
3804,3Sy6wgTNEp6vSgLs4EyXAf,Jóhann Jóhannsson;Yuki Numata Resnick;Tarn Tra...,Orphée,Flight from the City,ambient
20531,0Iv6ig3ixIwSbFHUCkHRVG,Skyy,Start Of A Romance,Start of a Romance,disco
979,1NF3eG2IpxKzR3jqMdR60O,Criolo,Espiral de Ilusão,Hora da Decisão,afrobeat
25520,6RCT4geaXTmDmtiwFLJv8j,Booka Shade;Eli & Fur;Einmusik,To the Sea,To the Sea - Einmusik Remix,electronic
18771,3uvALUobqjZvTGAh6737Mb,Kraak & Smaak;Parcels,Juicy Fruit,Stumble,deep-house
62596,1nW7N99dQBxlSXdRfYIoHZ,Parquet Courts,Human Performance,Pathos Prairie,psych-rock
27893,79Gu9WpaqKsuAVX7BF0zwb,Pablo,Um Novo Passo,Registro de Amor,forro
33594,3hLpuwMvMO1PQOPTNPbVjJ,disrupt,Unrest,Give It Back,grindcore


## Recommend

In [55]:
# Calculate input profile
input_profile = input_tracks.mean(axis=0).to_frame().T

In [56]:
# Find 20 neighbors
distances, indices = model.kneighbors(input_profile, n_neighbors=20)

# The model returns an array of indices
# Return the tracks from the original dataframe
recommendations = df.iloc[indices[0]]

recommendations[['track_id', 'artists', 'album_name', 'track_name', 'track_genre']]

,track_id,artists,album_name,track_name,track_genre
44230,4dWCaXMLTompf68X8srFF5,Lights Of Euphoria,Sleepwalk (The Awakening),Wicked Game (World on Fire Mix),industrial
20032,0w8tsAvPkiVQTiM7uRwM56,Theo Parrish,"Sound Signature Sounds, Vol. 2",Capritarious,detroit-techno
78126,1G0FRA9ymYcLSsCfR9Fqmz,Esthero,Breath From Another,Indigo Boy,trip-hop
28965,16hV7KbASg4EWoEymJerRA,NSH,Sunbeams,Sunbeams,french
75743,6DeDsUsG32QtXJiUGXLvNW,Indochine;John Digweed & Nick Muir,13,Henry Darger - John Digweed & Nick Muir Remix,synth-pop
75215,1WM80A5a4xDtlndjqjZQIv,Chromatics,Girls Just Wanna Have Fun,Girls Just Wanna Have Some,synth-pop
10573,1FDthLE4j0WhCPHSOZRnuM,Vivian Lai,真經典 - 黎瑞恩,發誓,cantopop
55705,5RHcBbi9TvOKvU5TUpeFa9,Enigma,Seven Lives Many Faces,The Same Parents,new-age
1627,2Szjsv5SLiGyCPAraUnNvk,Gyedu-Blay Ambolley,Ketan,Ketan,afrobeat
70346,1ODitWwCi2P2D489FX5CmY,Jasleen Royal,Jahaan Tu Chala - Midnight Mix (Gully Boy),Jahaan Tu Chala - Midnight Mix (Gully Boy),singer-songwriter


# Alternative models
- Model: NearestNeighbors with Manhattan distance
- Input tracks: average of multimodal preferences (Multi-seed k-NN)